### Advanced Model ChatBundestag

Advanced model identifies all suitable markers for metadata retrieval in user query

In [1]:
import os
import json

# data handling & viz
import pandas as pd
import matplotlib.pyplot as plt

# language preprocessing
import re #regex
from wordcloud import WordCloud
import spacy # DE stopwords


# langchain packages
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy

from langchain_core.documents import Document
#from langchain_core.runnables import RunnablePassthrough #may not be needed due to QA removal
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.embeddings import Embeddings
#from langchain_core.output_parsers import StrOutputParser appears to be buggy

from langchain_groq import ChatGroq
from langchain_classic import hub # alternative to 'from langchain import hub', because this gave errors


from pydantic import BaseModel, ValidationError
from typing import Optional, Literal, List


# environment variables
from dotenv import load_dotenv
import warnings

load_dotenv()
warnings.filterwarnings('ignore')

# -- LLM -- instantiate ChatGroq with llama
llm = ChatGroq(
    model= "llama-3.1-8b-instant",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)


# -- Data -- refs for vector db handling
vector_db_name = "debates_lp19"
vector_db_path = f"vector_databases/vector_db_{vector_db_name}"


/Users/helge/Documents/Bootcamp_Repos/capstone_project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# cleaned and reduced data set
df_exp_debates = pd.read_csv("data/debates_lp19.csv")

In [3]:
df_exp_debates.shape

(26869, 11)

### Chunking

In [4]:
# chunk size based on document size
TINY_DOC_THRESHOLD   = 50    # e.g. interjections
SMALL_DOC_THRESHOLD  = 400   # short statements = 1 chunk
MEDIUM_DOC_THRESHOLD = 1500  # medium speeches = paragraph-level split
# anything above: full recursive split with overlap

CHUNK_SIZE_MEDIUM = 500      # characters
CHUNK_SIZE_LARGE  = 400      # characters
CHUNK_OVERLAP     = 50       # roughly 10%

BATCH_SIZE = 500             # reduce to 250 if RAM issues


In [5]:
# split text
# todo: hard speech split to mark beginning and end of MPs speech (document)
def get_splitter(chunk_size):
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ".", ";", ","], 
       
        
    )

paragraph_splitter = get_splitter(CHUNK_SIZE_MEDIUM)
token_splitter     = get_splitter(CHUNK_SIZE_LARGE)


In [6]:
# dynamic chunking (document-aware)
def chunk_single_document(doc: Document, doc_idx: int) -> list[Document]:
    """
    Applies dynamic chunking based on document length.
    Never merges across documents.

    Size buckets (characters):
        tiny   < 50     : 1 chunk, flagged (low semantic signal)
        small  < 400    : 1 chunk
        medium < 1500   : paragraph-level split
        large  ≥ 1500   : recursive token-level split with overlap
    """
    text = doc.page_content.strip()

    if not text:
        return []

    length = len(text)

    # Determine strategy
    if length < TINY_DOC_THRESHOLD:
        strategy = "tiny"
        splits = [doc]

    elif length < SMALL_DOC_THRESHOLD:
        strategy = "small"
        splits = [doc]

    elif length < MEDIUM_DOC_THRESHOLD:
        strategy = "medium"
        splits = paragraph_splitter.split_documents([doc])

    else:
        strategy = "large"
        splits = token_splitter.split_documents([doc])

    # attach metadata to every chunk
    total = len(splits)
    for i, chunk in enumerate(splits):
        chunk.metadata.update({
            "chunk_id":          f"doc{doc_idx}_chunk{i}",
            "chunk_index":       i,
            "total_chunks":      total,
            "is_full_document":  total == 1,
            "chunking_strategy": strategy,
            "doc_char_length":   length,
        })

    return splits


In [10]:
# batch pipeline
def df_to_document_batches(df, batch_size=BATCH_SIZE):
    """Generator: yields batches of Documents without loading full df into memory."""
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start + batch_size]
        yield [
            Document(
                page_content=row['text'],
                metadata={
                    'row_index':        i,
                    'speaker_name':     row['speech_identification_ent'],
                    'date':             row['date'],
                    'year': str(pd.to_datetime(row['date']).year), # new column with year from date column
                    'legislative_period': row['period'],
                    'session':          row['session'],
                    'role':             row['Role'],
                    'governing_party':  row['governing_Party'],
                    'party':            row['Party'],
                }
            )
            for i, row in batch.iterrows()
        ]

In [11]:
# run chunking pipeline
def run_chunking_pipeline(df, batch_size=BATCH_SIZE) -> list[Document]:
    """
    Full batched chunking pipeline.
    Tracks strategy distribution so you can inspect and tune thresholds.
    """
    all_chunks = []
    strategy_counts = {"tiny": 0, "small": 0, "medium": 0, "large": 0}
    total_docs = 0

    for batch_idx, doc_batch in enumerate(df_to_document_batches(df, batch_size)):
        for local_idx, doc in enumerate(doc_batch):
            global_idx = batch_idx * batch_size + local_idx
            chunks = chunk_single_document(doc, global_idx)
            all_chunks.extend(chunks)

            if chunks:
                strategy_counts[chunks[0].metadata["chunking_strategy"]] += 1

        total_docs += len(doc_batch)
        print(f"Batch {batch_idx + 1}: {total_docs} docs → {len(all_chunks)} chunks")

    print(f"\nChunking-Strategie: {strategy_counts}")
    print(f"Gesamt: {len(all_chunks)} Chunks aus {total_docs} Dokumenten")
    return all_chunks

In [12]:
# chunking
chunks = run_chunking_pipeline(df_exp_debates)

Batch 1: 500 docs → 6254 chunks
Batch 2: 1000 docs → 12357 chunks
Batch 3: 1500 docs → 18494 chunks
Batch 4: 2000 docs → 25388 chunks
Batch 5: 2500 docs → 33217 chunks
Batch 6: 3000 docs → 40820 chunks
Batch 7: 3500 docs → 46900 chunks
Batch 8: 4000 docs → 55160 chunks
Batch 9: 4500 docs → 63264 chunks
Batch 10: 5000 docs → 69676 chunks
Batch 11: 5500 docs → 76077 chunks
Batch 12: 6000 docs → 84460 chunks
Batch 13: 6500 docs → 90807 chunks
Batch 14: 7000 docs → 97241 chunks
Batch 15: 7500 docs → 103525 chunks
Batch 16: 8000 docs → 110116 chunks
Batch 17: 8500 docs → 115941 chunks
Batch 18: 9000 docs → 122754 chunks
Batch 19: 9500 docs → 128457 chunks
Batch 20: 10000 docs → 135038 chunks
Batch 21: 10500 docs → 141663 chunks
Batch 22: 11000 docs → 148356 chunks
Batch 23: 11500 docs → 156634 chunks
Batch 24: 12000 docs → 163146 chunks
Batch 25: 12500 docs → 169633 chunks
Batch 26: 13000 docs → 176123 chunks
Batch 27: 13500 docs → 184725 chunks
Batch 28: 14000 docs → 190722 chunks
Batch 29

In [13]:
# display results of chunking function
print(f"number of chunks created: {len(chunks)}")

number of chunks created: 356390


### Embedding

In [14]:
# instantiate embedding model 
embedding = HuggingFaceEmbeddings(
        model_name='intfloat/multilingual-e5-small', 
        # cloud upgrade path: intfloat/multilingual-e5-base or deepset/gbert-base or BAAI/bge-m3
        model_kwargs={'device': 'mps'}, 
        # cloud: 'mps' -> 'cuda'.
        encode_kwargs={"normalize_embeddings": True,
                      "prompt": "passage: "  # adds prefix automatically at indexing time
                      }
    )

In [15]:
# prefix handling for embedding model
class E5QueryWrapper(Embeddings):
    """
    Wraps HuggingFaceEmbeddings to add the required 'query: ' prefix
    for multilingual-e5 models at retrieval time only.
    """
    def __init__(self, base_embedding: Embeddings):
        self.base = base_embedding

    def embed_query(self, text: str) -> List[float]:
        return self.base.embed_query(f"query: {text}")

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # prefix 'passage: ' already handled in encode_kwargs at indexing time
        return self.base.embed_documents(texts)

embedding_wrapped = E5QueryWrapper(embedding)

In [16]:
# function to create and save vector store 
def create_and_store(chunks,vector_db_path,embedding):
    """
    this function creates a vector store from chunks and saves it locally
    """
    # create the vector store 
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embedding, # parameter name is singular!
        distance_strategy=DistanceStrategy.COSINE  # or DistanceStrategy.DOT or DistanceStrategy.L2 
    )
    
     # save vector store locally
    vectorstore.save_local(vector_db_path)

    return vectorstore

In [17]:
# implement retrieval from FAISS db

def retrieve_from_vector_db(vector_db_path,embedding):
    """
    this function splits out a retriever object from a local vector store
    """
    
    vectorstore = FAISS.load_local(
        folder_path=vector_db_path,
        embeddings=embedding, # parameter name is plural!
        allow_dangerous_deserialization=True,
        distance_strategy=DistanceStrategy.COSINE  # or DistanceStrategy.DOT or DistanceStrategy.L2 
    )
    retriever = vectorstore.as_retriever(
        search_kwargs={
            'k':15, # k nearest
        }  
    )
    
    return retriever,vectorstore


In [18]:
# check if vector store exists. if no: creates vector store
if not os.path.exists(vector_db_path):
        print("Vector DB not found. Creating and embedding chunks.")
        vectorstore=create_and_store(chunks=chunks, vector_db_path=vector_db_path, embedding=embedding_wrapped)
        print(f"Vector DB save to {vector_db_path}")
else:
    print(f"Vector DB found at {vector_db_path}. Skipping embedding.")

Vector DB not found. Creating and embedding chunks.
Vector DB save to vector_databases/vector_db_debates_lp19


In [20]:
# load the retriever and index
retriever,vectorstore = retrieve_from_vector_db(vector_db_path=vector_db_path, embedding=embedding_wrapped)

#type(retriever),type(vectorstore)

### Prompt

In [26]:
# ── [8] Filter map ────────────────────────────────────────────────────────────
# Maps natural-language user terms → (metadata_key, metadata_value).
# Extend here when adding new filter terms - no changes needed elsewhere.

FILTER_MAP = {
    # parties
    "spd":       ("Party", "SPD"),
    "cdu":       ("Party", "CDU"),
    "csu":       ("Party", "CSU"),
    "grüne":     ("Party", "Grüne"),
    "fdp":       ("Party", "FDP"),
    "linke":     ("Party", "Linke"),
    "afd":       ("Party", "AfD"),
    "fraktionslos": ("Party", "fraktionslos"),
    "parteilos": ("Party", "parteilos"),
    # -- add other parties to complete list since 1949
    
    # government status
    "kabinett":       ("Party", "Cabinet"),   # governing_Party == Cabinet
    "regierungspartei": ("governing_Party", 1),
    "opposition":       ("governing_Party", 0),
    
    # roles
    "kanzler":         ("Role", "Bundeskanzler"),
    "bundeskanzler":   ("Role", "Bundeskanzler"),
    "kanzlerin":       ("Role", "Bundeskanzler"),
    "bundeskanzlerin": ("Role", "Bundeskanzler"),
    "minister":        ("Role", "Bundesminister"),
    "bundesminister":  ("Role", "Bundesminister"),
    "staatssekretär":  ("Role", "Staatssekretär"),
    "staatsminister":  ("Role", "Staatsminister"),
    "abgeordnete":     ("Role", "MdB"),
    "mdb":             ("Role", "MdB"),
    "mitglied des bundestags":("Role", "MdB"),
    
    # legislative periods
    "19. wahlperiode": ("period", "19"),
    "19 wahlperiode":  ("period", "19"),
    "wp19":            ("period", "19"),
    "18. wahlperiode": ("period", "18"),
    "18 wahlperiode":  ("period", "18"),
    "wp18":            ("period", "18"),
    
    # time frame 
    "2021": ("year", "2021"),
    "2020": ("year", "2020"),
    "2019": ("year", "2019"),
    "2018": ("year", "2018"),
    "2017": ("year", "2017"),
    # -- add remaining time frames til 1949 for full data set
}

In [27]:
# extract metadata filters and semantic query from free-text user input 
# handles: party, role, governing_party, cabinet, speaker, session, date, legislative_period
# returns (semantic_query, filters) tuple
def parse_query_filters(user_input: str) -> tuple[str, dict]:
    """
    Parses a free-text user query into a semantic search string and a
    metadata filter dict for FAISS.

    Extraction order (all additive, multiple filters supported):
      1. Party affiliation     — matched via FILTER_MAP
      2. Government status     — Kabinett / Regierungspartei / Opposition
      3. Role                  — Minister, Staatssekretär, Abgeordnete
      4. Legislative period    — "19. Wahlperiode", "WP19", etc.
      5. Session               — "Sitzung 42", "42. Sitzung"
      6. Date                  — ISO (2019-03-14) or German (14.03.2019)
      7. Speaker name          — "von [Name]" or "Redner [Name]"

    Returns:
        semantic_query : str   — residual text after filter terms removed
        filters        : dict  — metadata key/value pairs for FAISS filter
    """
    filters = {}
    semantic = user_input.lower()

    # 1–4: FILTER_MAP lookup
    for term, (key, value) in FILTER_MAP.items():
        if term in semantic:
            filters[key] = value
            semantic = semantic.replace(term, "")

    # 5: Session — "Sitzung 42" or "42. Sitzung"
    session_match = re.search(r'(\d+)\.\s*sitzung|sitzung\s*(\d+)', semantic)
    if session_match:
        session_num = session_match.group(1) or session_match.group(2)
        filters["session"] = session_num
        semantic = re.sub(r'(\d+)\.\s*sitzung|sitzung\s*(\d+)', '', semantic)

    # 6: Date — ISO (2019-03-14) or German (14.03.2019)
    date_match = re.search(r'\d{4}-\d{2}-\d{2}|\d{2}\.\d{2}\.\d{4}', semantic)
    if date_match:
        filters["date"] = date_match.group()
        semantic = semantic.replace(date_match.group(), "")

    # 7: Speaker — "von Müller" or "Redner Müller"
    speaker_match = re.search(r'(?:von|redner[in]?)\s+([A-ZÄÖÜ][a-zäöüß\-]+)', user_input)
    if speaker_match:
        filters["speaker_name"] = speaker_match.group(1)
        semantic = re.sub(
            r'(?:von|redner[in]?)\s+' + re.escape(speaker_match.group(1)),
            '', semantic, flags=re.IGNORECASE
        )

    semantic = re.sub(r'\s+', ' ', semantic).strip()
    return semantic, filters

In [28]:
# inject chunk metadata into context string (-> LLM)
# attributes extracted arguments to specific speakers/ sessions
def format_context_with_metadata(docs: list[Document]) -> str:
    """
    Prepends a metadata header to each retrieved chunk so the LLM can
    attribute extracted arguments to specific speakers, parties and sessions.
    Fields with missing/NaN values are omitted from the header.
    """
    formatted = []
    for doc in docs:
        m = doc.metadata
        header_parts = []
        if m.get("speaker_name"): header_parts.append(f"Redner: {m['speaker_name']}")
        if m.get("party"):        header_parts.append(f"Partei: {m['party']}")
        if m.get("role"):         header_parts.append(f"Rolle: {m['role']}")
        if m.get("date"):         header_parts.append(f"Datum: {m['date']}")
        if m.get("session"):      header_parts.append(f"Sitzung: {m['session']}")
        if m.get("legislative_period"):
            header_parts.append(f"Wahlperiode: {m['legislative_period']}")
        header = f"[{' | '.join(header_parts)}]" if header_parts else ""
        formatted.append(f"{header}\n{doc.page_content}".strip())
    return "\n\n---\n\n".join(formatted)

In [29]:
# retriever with dynamic metadata filters. Falls back to unfiltered retriever if no filters detected
def get_filtered_retriever(vectorstore, filters: dict, k: int = 15):
    """
    Returns a retriever with metadata filters applied.
    Falls back to unfiltered retrieval if filters dict is empty.
    """
    search_kwargs = {"k": k}
    if filters:
        search_kwargs["filter"] = filters
    return vectorstore.as_retriever(search_kwargs=search_kwargs)

In [30]:
# prompt
prompt = ChatPromptTemplate.from_template(
    """
    Du bist ein Assistent für Argumentationsanalyse, spezialisiert auf parlamentarische Debatten des Deutschen Bundestags.

    Deine Aufgabe ist es, die Argumentationsstruktur aus dem untenstehenden Redeauszug zu extrahieren,
    wobei du die Frage des Nutzers als thematischen Fokus verwendest.

    Definitionen:
    - claim: die zentrale politische Position oder der Vorschlag, für den oder gegen den argumentiert wird
    - supporting_premises: Aussagen, Fakten oder Gründe, die den claim stützen
    - opposing_premises: Gegenargumente, Einwände oder Positionen, die abgelehnt werden

    Regeln:
    - Verwende NUR Informationen aus dem Kontext, keine externen Kenntnisse
    - Zitiere oder paraphrasiere eng aus dem Text, nichts erfinden
    - Fokussiere die Extraktion auf die Frage des Nutzers, ignoriere thematisch irrelevante Textteile
    - Entnimm Redner, Partei, Rolle, Datum, Sitzung und Wahlperiode aus den Metadaten-Kopfzeilen im Kontext
    - Setze Felder auf null, wenn die entsprechenden Informationen im Kontext nicht vorhanden sind
    - Wenn kein klarer claim erkennbar ist, setze "claim" auf null
    - Wenn keine supporting_premises vorhanden sind, gib eine leere Liste zurück
    - Wenn keine opposing_premises vorhanden sind, gib eine leere Liste zurück
    - Gib NUR valides JSON zurück, keine Erklärung, keine Einleitung
    - Wenn der Kontext keine relevanten Informationen zur Anfrage enthält, gib folgendes zurück:
        {{"claim": null, "supporting_premises": [], "opposing_premises": [], 
        "confidence": "low", "note": "Keine relevanten Informationen gefunden. 
        Versuche die Suchanfrage zu vereinfachen oder Filter zu reduzieren."}}

    Beispiel Frage: "Welche Position vertritt Svenja Schulze zur CO2-Bepreisung?"

    Beispiel Kontext:
    [Redner: Svenja Schulze | Partei: SPD | Rolle: Bundesministerin für Umwelt, Naturschutz und nukleare Sicherheit | Datum: 2019-06-05 | Sitzung: 103 | Wahlperiode: 19]
    Wir brauchen eine CO2-Bepreisung, weil sie das teuer macht, was wir vermeiden wollen. Sie kann eine Lenkungswirkung entfalten. 
    Sie ist damit ein wichtiges Element in einer Gesamtstrategie zum Klimaschutz. Aber sie löst nicht alle Probleme. Ein CO2-Preis, 
    der alle anderen Instrumente überflüssig macht, würde bedeuten, dass man sehr hoch einsteigen müsste, und das kann dann nicht mehr sozial gerecht sein. 
    Ein CO2-Preis muss aber fair sein. Der Staat muss die Einnahmen an die Bürgerinnen und Bürger zurückgeben; 
    denn vor allem Pendler, vor allen Dingen Mieterinnen und Mieter können dem höheren Preis durch ihr eigenes Verhalten oft gar nicht ausweichen.

    Beispiel Output:
    {{
      "claim": "Klimaschutz ist die zentrale Gestaltungsaufgabe dieser Regierung in 2019",
      "supporting_premises": [
        "Wissenschaftlerinnen und Wissenschaftler weltweit belegen den menschengemachten Klimawandel",
        "Mit dem Kohleausstieg wurde ein sozial- und klimaverträglicher Konsens erreicht",
        "Ein Klimaschutzgesetz liegt vor, das verbindliche CO2-Einsparziele für alle Sektoren regeln soll"
        ],
        
      "opposing_premises": [
        "Eine Renaissance der Atomkraft wird als Lösung ins Spiel gebracht",
        "Ein sehr hoher CO2-Preis als alleiniges Instrument sei die Lösung"
        ],
        
      "speaker": "Svenja Schulze",
      "party": "SPD",
      "role": "Bundesministerin für Umwelt, Naturschutz und nukleare Sicherheit",
      "date": "2019-06-05",
      "session": "103",
      "legislative_period": "19",
      "confidence": "high",
      "reasoning": "Der claim ist ein direktes Zitat aus der Rede ('Klimaschutz ist die zentrale Gestaltungsaufgabe dieser Regierung in 2019'). Die supporting_premises sind explizit genannte Begründungen im Redetext. Die opposing_premises entsprechen zwei Positionen, die Schulze im Text direkt benennt und widerlegt: die Atomkraft-Renaissance und einen allein wirkenden hohen CO2-Preis."
    }}

    Kontext:
    {context}

    Frage:
    {question}
    """)

In [38]:
# pydantic schema for output
# todo: revise optional attributes
# todo: update toulmin fields
class ArgumentStructure(BaseModel):
    # Core argument
    claim:               Optional[str]
    supporting_premises: List[str]
    opposing_premises:   List[str]
    # Speaker attribution
    speaker:             Optional[str]
    party:               Optional[str]
    role:                Optional[str]
    date:                Optional[str]
    session:             Optional[str]
    legislative_period:  Optional[str]
    # Quality
    confidence:          Literal["high", "medium", "low"] 
    reasoning:           Optional[str] = None
    note:                Optional[str] = None # carries message if no result
    # Extended Toulmin fields — v2, inactive for now
    # warrant:   Optional[str] = None   # inference rule connecting premises to claim
    # backing:   Optional[str] = None   # support for the warrant itself
    # rebuttal:  Optional[str] = None   # conditions under which claim does not hold



In [39]:
# check and format LLM output: according to Pydantic schema
def parse_llm_output(raw_output: str) -> dict:
    """
    Cleans, parses and validates LLM output against ArgumentStructure schema.
    Returns {"status": "ok", "data": ...} or error dict with raw output for debugging.
    """
    try:
        cleaned = re.sub(r"```json|```", "", raw_output).strip()
        parsed = json.loads(cleaned)
        validated = ArgumentStructure(**parsed)
        return {"status": "ok", "data": validated.model_dump()}
    except json.JSONDecodeError as e:
        return {"status": "json_error",        "raw": raw_output, "error": str(e)}
    except ValidationError as e:
        return {"status": "validation_error",  "raw": raw_output, "error": str(e)}


In [40]:
# runnable RAG pipeline (LCEL)
# parse user input -> retrieve
def connect_chains(retriever, vectorstore):
    """
    Full RAG chain with dynamic metadata filtering.
    Connects retriever, prompt and llm into a RAG chain for argument extraction.
    User question both drives semantic search and focuses the extraction.

    Flow:
      1. Parse user input → (semantic_query, filters)
      2. Build filtered retriever from vectorstore
      3. Retrieve top-k chunks matching semantic query + metadata filters
      4. Format chunks with metadata headers
      5. Pass formatted context + original question to prompt
      6. LLM generates structured argument JSON
      7. Parse and validate output against ArgumentStructure schema
    """
    def retrieve_and_format(user_input: str) -> dict:
        semantic_query, filters = parse_query_filters(user_input) #extract metadata filters and semantic query from user input
        filtered_retriever = get_filtered_retriever(vectorstore, filters) #retriever applying meta data filters
        docs = filtered_retriever.invoke(semantic_query)
        return {
            "context":  format_context_with_metadata(docs),
            "question": user_input   # original question — preserves full user intent for LLM
        }

    rag_chain = (
        RunnableLambda(retrieve_and_format)
        | prompt
        | llm
        | RunnableLambda(lambda msg: parse_llm_output(msg.content))
    )
    return rag_chain

In [41]:
# pass vectorstore as second argument
rag_chain = connect_chains(retriever, vectorstore)

In [42]:
# chat loop
ATTRIBUTION_FIELDS = [
    ("speaker",            "👤 Redner"),
    ("party",              "🏛️ Partei"),
    ("role",               "💼 Rolle"),
    ("date",               "📅 Datum"),
    ("session",            "📋 Sitzung"),
    ("legislative_period", "🗓️ Wahlperiode"),
]


In [43]:
# rag chain            
def chat_with_rag(chain):
    """
    Interactive RAG chat loop with argument mining output.
    Handles: structured output, no-results case, JSON errors.
    """
    print("Willkommen im ChatBundestag 🏛️. Gib eine Frage zu den Bundestagsdebatten ein.")
    print("Schreibe 'exit', um den Chat zu beenden.\n")

    while True:
        user_input = input("Du: ").strip()

        if not user_input:
            continue # skip accidental empty enters to prevent crash

        if user_input.lower() in ["exit", "quit"]:
            print("Chat wird beendet. Auf Wiedersehen!")
            break

        try:
            result = chain.invoke(user_input)

            # ── JSON / validation error ──────────────────────────────────────
            if result["status"] != "ok":
                print(f"\n⚠️  Parsing fehlgeschlagen ({result['status']})")
                print(f"Fehler: {result['error']}")
                print(f"Rohausgabe:\n{result['raw']}\n") 
                continue

            data = result["data"]

            # ── No relevant results ──────────────────────────────────────────
            if data.get("note"):          
                print(f"\n⚠️  {data['note']}\n") 
                continue

            if not data["claim"]:
                print("\n⚠️  Keine relevanten Informationen gefunden.")
                print("Tipp: Versuche die Anfrage zu vereinfachen oder weniger Filter zu verwenden.\n")
                continue

            # ── Normal structured output ─────────────────────────────────────
            print("\n" + "─" * 60)

            # Speaker / context info
            speaker   = data.get("speaker")   or "unbekannt"
            party     = data.get("party")      or "?"
            role      = data.get("role")       or "?"
            date      = data.get("date")       or "?"
            period    = data.get("legislative_period") or "?"
            confidence = data.get("confidence") or "?"

            print(f"👤  {speaker} ({party} · {role})")
            print(f"📅  {date}  |  Legislaturperiode: {period}")
            print(f"🎯  Konfidenz: {confidence}")
            print("─" * 60)

            # Claim
            print(f"\n📌  Claim:\n    {data['claim']}\n")

            # Supporting premises
            supporting = data.get("supporting_premises", [])
            if supporting:
                print("✅  Stützende Prämissen:")
                for p in supporting:
                    print(f"    • {p}")
            else:
                print("✅  Stützende Prämissen: keine gefunden")

            # Opposing premises
            opposing = data.get("opposing_premises", [])
            if opposing:
                print("\n❌  Gegenargumente:")
                for p in opposing:
                    print(f"    • {p}")
            else:
                print("\n❌  Gegenargumente: keine gefunden")

            print("─" * 60 + "\n")

        except Exception as e:
            print(f"\n❌  Fehler: {e}\n")
            


In [ ]:
# run chat
chat_with_rag(rag_chain)

Willkommen im ChatBundestag 🏛️. Gib eine Frage zu den Bundestagsdebatten ein.
Schreibe 'exit', um den Chat zu beenden.



Du:  Was sind die Argumente der Grünen für das Klimaschutzgesetz?



⚠️  Keine expliziten Argumente der Grünen für das Klimaschutzgesetz gefunden. Die Rede bezieht sich auf die Position der Grünen nicht explizit.



Du:  Wie steht die FDP zur Mietpreisbremse?



⚠️  Keine expliziten Aussagen der FDP zur Mietpreisbremse gefunden. Die Auswertung basiert auf allgemeinen Positionen der FDP.



...


⚠️  Keine expliziten Aussagen der FDP zur Mietpreisbremse gefunden. Die Auswertung basiert auf allgemeinen Positionen der FDP.



Du:  Wie steht die Regierung zur Mietpreisbremse?



⚠️  Parsing fehlgeschlagen (validation_error)
Fehler: 6 validation errors for ArgumentStructure
speaker
  Field required [type=missing, input_value={'claim': None, 'supporti... Filter zu reduzieren.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
party
  Field required [type=missing, input_value={'claim': None, 'supporti... Filter zu reduzieren.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
role
  Field required [type=missing, input_value={'claim': None, 'supporti... Filter zu reduzieren.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
date
  Field required [type=missing, input_value={'claim': None, 'supporti... Filter zu reduzieren.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
session
  Field required [type=missing, input_value={'claim': None, 'supporti... Filter zu reduzieren.'}, input

Du:  Welche Position vertritt Gerd Müller zur Entwicklungspolitik?



⚠️  Parsing fehlgeschlagen (validation_error)
Fehler: 6 validation errors for ArgumentStructure
speaker
  Field required [type=missing, input_value={'claim': None, 'supporti... Filter zu reduzieren.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
party
  Field required [type=missing, input_value={'claim': None, 'supporti... Filter zu reduzieren.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
role
  Field required [type=missing, input_value={'claim': None, 'supporti... Filter zu reduzieren.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
date
  Field required [type=missing, input_value={'claim': None, 'supporti... Filter zu reduzieren.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
session
  Field required [type=missing, input_value={'claim': None, 'supporti... Filter zu reduzieren.'}, input

### error log
- always identify speaker, party and date
- don't generate output if query is unclear/ ill-formulated / lacking essential components
- prompt does not take into account {reasoning} for now
- how is the topic introduced for which to retrieve claim/premises ? Query should filter the database preemptively - so query + context in prompt!
- '\xa0–' as part of output
- validation errors for ArgumentStructure: missing confidence, reasoning ("Welche Position vertritt Svenja Schulze zur CO2-Bepreisung?")

### erroneous query log
- Wie steht die FDP zur Mietpreisbremse? (identical output for supporting / opposing)
- Wie steht die Regierung zur Mietpreisbremse? (error 413)
- Wie steht die CDU zum Brexit? (json_error, Parsing fehlgeschlagen)
- Was sagt Angela Merkel zur Covid-19 Situation? (wrong extraction, query does not respect pre-defined format)
- Was sind Angela Merkels Argumente für Corona-Hilfen? (no output whatsoever)
- Was sind die Argumente der Grünen für das Klimaschutzgesetz? (OK output for V0)
- Finde Merkels Argumente zur Maskenaffäre (characters out of place)
- Wie steht Angela Merkel zum Thema Maskenpflicht? (output empty)


### expected query log
- Welche Position vertritt Svenja Schulze zur CO2-Bepreisung ? 

